## Matrix-free rSVD adjoint
This notebook tests out the improved implementation of the `MatrixFreeRSVD` and the new `MatrixFreeRSVDAdjoint` solvers for approximating the SVD of $K$ and $K^*$, respectively.

---

In [ ]:
import numpy as np
from fenics import UnitSquareMesh, FunctionSpace

/home/elias/miniforge3/envs/fenics_env/lib/python3.9/site-packages/ufl/__init__.py:250: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


First, I do a simple check to see the improvement in the new `MatrixFreeRSVD`:

In [2]:
from algorithms.matrix_free_rsvd import MatrixFreeRSVD as MatrixFreeRSVDOld
from algorithms.rsvd_solvers import MatrixFreeRSVD


# Function space
n = 64
mesh = UnitSquareMesh(n, n)
V_h = FunctionSpace(mesh, 'CG', 1)

rsvd = MatrixFreeRSVD(V_h)
rsvd_old = MatrixFreeRSVDOld(V_h)

In [12]:
%timeit -n 20 -r 10 MatrixFreeRSVD(V_h, precompute='LU').solve(k=25)

228 ms ± 9.63 ms per loop (mean ± std. dev. of 10 runs, 20 loops each)


In [13]:
%timeit -n 20 -r 10 MatrixFreeRSVD(V_h, precompute='Cholesky').solve(k=25)

205 ms ± 11.7 ms per loop (mean ± std. dev. of 10 runs, 20 loops each)


In [14]:
%timeit -n 20 -r 10 MatrixFreeRSVDOld(V_h).mf_rsvd(k=25)

430 ms ± 7.78 ms per loop (mean ± std. dev. of 10 runs, 20 loops each)


The new implementation is approximately twice as fast on this problem size.

Next, I check that the output of `MatrixFreeRSVDAdjoint` is correct:

In [ ]:
from utils.exact_forward_operator import ExactForwardOperator
from algorithms.rsvd_solvers import MatrixFreeRSVDAdjoint

k = 50

# Get exact K for reference
efo = ExactForwardOperator(V_h)
K_exact = efo.K
M = efo.M_dx
M_ds = efo.M_ds

# Run both solvers
rsvd_adj = MatrixFreeRSVDAdjoint(V_h)
U, S, Vt = rsvd_adj.solve(k)

# Reconstruct K: K ≈ M_∂⁻¹ V S Uᵀ M
K_reconstructed = rsvd_adj._solve_M_ds(Vt.T) @ np.diag(S) @ (M @ U)

print(np.linalg.norm(K_exact - K_reconstructed, 'fro') / np.linalg.norm(K_exact, 'fro'))

0.00865188431589624


The error is within expected, so the implementation is correct.